### Importing Dependencies

In [1]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import os

c:\Users\harsh\OneDrive\Desktop\Custom_RAG\cragenv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\harsh\OneDrive\Desktop\Custom_RAG\cragenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Data Ingestion

In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader


def data_processing(pdf_file_path: str):
    """
    1. Accept a single PDF file path as input.
    2. Load the PDF content.
    3. Store all pages as documents in a single list.
    4. Add metadata (source filename and file type).
    """
    all_documents = []
    pdf_path = Path(pdf_file_path)

    # Check if file exists
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF file not found: {pdf_file_path}")

    print(f"Processing: {pdf_path.name}")

    try:
        loader = PyMuPDFLoader(str(pdf_path))
        documents = loader.load()

        for doc in documents:
            doc.metadata["source_file"] = pdf_path.name
            doc.metadata["file_type"] = "pdf"
            all_documents.append(doc)
    except Exception as e:
        print(f"Error processing {pdf_path.name}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Example usage (user gives a single PDF file)
pdf_file = r"C:\Users\harsh\OneDrive\Desktop\Custom_RAG\DogTraining101.pdf"  # ← user input goes here
all_pdf_documents = data_processing(pdf_file)

Processing: DogTraining101.pdf

Total documents loaded: 244


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'creationdate': '2018-02-26T12:49:28-05:00', 'source': 'C:\\Users\\harsh\\OneDrive\\Desktop\\Custom_RAG\\DogTraining101.pdf', 'file_path': 'C:\\Users\\harsh\\OneDrive\\Desktop\\Custom_RAG\\DogTraining101.pdf', 'total_pages': 244, 'format': 'PDF 1.6', 'title': 'Dog Training 101', 'author': 'Jean Donaldson', 'subject': '', 'keywords': '', 'moddate': '2018-02-27T17:05:02-05:00', 'trapped': '', 'modDate': "D:20180227170502-05'00'", 'creationDate': "D:20180226124928-05'00'", 'page': 0, 'source_file': 'DogTraining101.pdf', 'file_type': 'pdf'}, page_content='Better Living\nTopic\nHobby & Leisure\nSubtopic\nDog Training 101\nJean Donaldson\nThe Academy for Dog Trainers\nCourse Guidebook'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'creationdate': '2018-02-26T12:49:28-05:00', 'source': 'C:\\Users\\harsh\\OneDrive\\Desktop\\Cu

### Chunking

In [4]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Splitting the documents into small chunks for better RAG performance.
    """

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators = ["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk

    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}....")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)

Split 244 documents into 458 chunks

 Example chunk:
Content: Better Living
Topic
Hobby & Leisure
Subtopic
Dog Training 101
Jean Donaldson
The Academy for Dog Trainers
Course Guidebook....
Metadata: {'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'creationdate': '2018-02-26T12:49:28-05:00', 'source': 'C:\\Users\\harsh\\OneDrive\\Desktop\\Custom_RAG\\DogTraining101.pdf', 'file_path': 'C:\\Users\\harsh\\OneDrive\\Desktop\\Custom_RAG\\DogTraining101.pdf', 'total_pages': 244, 'format': 'PDF 1.6', 'title': 'Dog Training 101', 'author': 'Jean Donaldson', 'subject': '', 'keywords': '', 'moddate': '2018-02-27T17:05:02-05:00', 'trapped': '', 'modDate': "D:20180227170502-05'00'", 'creationDate': "D:20180226124928-05'00'", 'page': 0, 'source_file': 'DogTraining101.pdf', 'file_type': 'pdf'}


### Embedding

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any ,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """
    1. Load the Embedding model
    2. Generate Embedding
    """
    def __init__(self, model_name:str="all-MiniLM-L6-v2"):
        """
        Intialize the embedding manager

        Args: model_name: Huggingface model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """
        load the SentenceTransformer model
        """
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully, Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts:List[str]) -> np.ndarray:
        """
        Generate embedding for a list of texts

        Args:
        texts: Liat of text strings to embedding

        Returns:
        numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts..")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## Initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1160.37it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully, Embedding dimension: 384


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [9]:
# chunks

In [10]:

# 1. Extract the raw text from your chunks (from Cell 6)
texts_to_embed = [chunk.page_content for chunk in chunks]

# 2. Generate the embeddings for these texts using your EmbeddingManager
print("Generating embeddings...")
document_embeddings = embedding_manager.generate_embeddings(texts_to_embed)

# 3. Add both the chunks and their new embeddings to your VectorStore!
print("\nAdding to Vector Store...")
vectorstore.add_documents(documents=chunks, embeddings=document_embeddings)

Generating embeddings...
Generating embeddings for 458 texts..


Batches: 100%|██████████| 15/15 [00:37<00:00,  2.52s/it]


Generated embeddings with shape: (458, 384)

Adding to Vector Store...
Adding 458 documents to vector store...
Successfully added 458 documents to vector store
Total documents in collection: 458


### Retriver Pipeline from VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("Tell me about why dogs get angry")

Retrieving documents for query: 'Tell me about why dogs get angry'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.16it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_bb84e6c3_156',
  'content': '79\nDog Training 101\nAssessing Your Dog’s Emotional State\na\na One of the first things taught to aspiring behavior counselors at the \nAcademy for Dog Trainers is a\xa0diagnostic watershed. Any time a\xa0dog is \npresented with a\xa0behavior problem, consider whether the dog is upset. \nThe reason this is so critical is that the approach to dogs who are upset \nis different from the approach to dogs who are not upset. An upset dog \nis worried, anxious, uncomfortable, or afraid. This can be manifested in \na\xa0few ways: fight, flight, or freezing. \na\na The dog might be frankly fearful, in which case he might cower or flee. \nIf he’s prevented from fleeing, he attempts to flee. He might yelp, whine, \nscream, or be silent. An upset dog might also be aggressive. He might \ngrowl, snarl, snap, and bite. He might do so as his first plan, or he might \ndo so if his first plan was flight and that is preempted by a\xa0leash, walls, \nor restraint

### RAG Pipeline- VectorDB To LLM Output Generation

In [18]:
import os
from dotenv import load_dotenv
load_dotenv()

# print(os.getenv("GROQ_API_KEY"))

True

In [17]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

In [19]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.1-8b-instant", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    

In [20]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: llama-3.1-8b-instant
Groq LLM initialized successfully!


In [22]:
## Get the context from the retriver and pass it to the LLM

rag_retriever.retrieve("What makes dogs angry.")

Retrieving documents for query: 'What makes dogs angry.'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.80it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_bb84e6c3_156',
  'content': '79\nDog Training 101\nAssessing Your Dog’s Emotional State\na\na One of the first things taught to aspiring behavior counselors at the \nAcademy for Dog Trainers is a\xa0diagnostic watershed. Any time a\xa0dog is \npresented with a\xa0behavior problem, consider whether the dog is upset. \nThe reason this is so critical is that the approach to dogs who are upset \nis different from the approach to dogs who are not upset. An upset dog \nis worried, anxious, uncomfortable, or afraid. This can be manifested in \na\xa0few ways: fight, flight, or freezing. \na\na The dog might be frankly fearful, in which case he might cower or flee. \nIf he’s prevented from fleeing, he attempts to flee. He might yelp, whine, \nscream, or be silent. An upset dog might also be aggressive. He might \ngrowl, snarl, snap, and bite. He might do so as his first plan, or he might \ndo so if his first plan was flight and that is preempted by a\xa0leash, walls, \nor restraint

### Integration Vectordb Context pipeline with LLM Output

In [23]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [24]:
answer=rag_simple("What makes dog angry",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What makes dog angry'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.38it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


According to the context, a dog might become angry if it is prevented from fleeing and attempts to flee is preempted by a leash, walls, or restraint.


### Enhanced RAG Pipeline Features

In [26]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Summaries to me about dog behaviour", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Summaries to me about dog behaviour'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.31it/s]

Generated embeddings with shape: (1, 384)


Retrieved 3 documents (after filtering)
Answer: Here's a concise summary of dog behavior based on the provided context:

**Key Points:**

1. Dogs have evolved to prioritize survival, including finding food, protecting themselves, and producing offspring.
2. Domestication has influenced dog behavior, easing pressure on certain traits while magnifying others.
3. Fight-or-flight behaviors are a crucial aspect of dog behavior, including defense of self and resources.
4. Conflicts can arise when dogs learn new behaviors, such as approach-approach and approach-avoidance conflicts, which can lead to barking.

**Common Conflicts:**

1. Approach-approach conflict: When a dog learns that moving towards a stimulus (e.g., a person) can lead to a reward, but their previous experience teaches them to move away from something.
2. Approach-avoidance conflict: When a dog is simultaneously attracted and repelled by a stimulus (e.g., a new horse), leading to oscillation between moving forward and retreat

In [28]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history


    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Summarize how to train dogs with minimal training", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Summarize how to train dogs with minimal training'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.26it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
13
Dog Training 101

181
Dog Training 101
behavior” by having him follow a food lure. Lurin

g well takes a bit of 
practice. This same lure can then be used as a reward, or something else 
can be used as the reward, depending on your strategy. 
mechanics • lesson 2
A trainer’s hands-on fluency, speed, and coordination are her mechanics. 
You may have encyclopedic knowledge of learning theory, all motivational 
ducks in a row, and a genius dog, but if your mechanics are not good, 
training will be less efficient. Because it is a procedural skill, the only 
way to improve mechanics is to train a lot. 
motivation • lesson 1
No properly functioning living organism will do something for nothing. 
And in spite of cinematic attempts to portray dogs as having a “desire 
to please,” dogs are not exempt, and require concrete motivation. If you 
operationalize the actions of trainers who make claims to the contrary, 
you find that they inevitably use motivators such as choke collars, pain,

Lesson 24
TRAINING 
CHALLENGES AND 
SOLUTIONS
A 
large part of being an effective trainer is prob